# The Four Scapes of Cinema — Analysis Notebook
**INFO 230 · Moe Alhassan · Spring 2026**

This notebook walks through the complete analysis pipeline for the film network disjuncture project. 
The interactive site presents the findings; this notebook shows how they were produced.

**Data sources:** IMDb Non-Commercial Datasets, TMDB API, Wikidata SPARQL

[→ View the interactive site](https://moealhassan.github.io/INFO230/MiniProject2/site-v2/)


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import json
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

DATA_PROCESSED = Path("../data/processed")
DATA_NETWORKS = Path("../data/networks")

# Load processed datasets
films = pd.read_parquet(DATA_PROCESSED / "films.parquet")
people = pd.read_parquet(DATA_PROCESSED / "people.parquet")
film_people = pd.read_parquet(DATA_PROCESSED / "film_people.parquet")
origins = pd.read_parquet(DATA_PROCESSED / "wikidata_origins.parquet")

print(f"Films: {len(films)}")
print(f"People: {len(people)}")
print(f"Film-person edges: {len(film_people)}")
print(f"People with country data: {origins['country'].notna().sum()} ({origins['country'].notna().mean()*100:.1f}%)")


## 2. Dataset Overview

The corpus contains the 1,778 most-voted movies on IMDb (153,000+ votes), spanning 1927–2025.


In [ ]:
# Year distribution
print("Year range:", int(films["startYear"].dropna().min()), "-", int(films["startYear"].dropna().max()))
print("Rating range:", films["averageRating"].min(), "-", films["averageRating"].max())
print(f"
Films with budget data: {films['budget'].notna().sum()} ({films['budget'].notna().mean()*100:.0f}%)")
print(f"
Top 10 genres:")
print(films["genres"].str.split(",").explode().value_counts().head(10))
print(f"
Role distribution:")
print(film_people["category"].value_counts())


In [ ]:
# Top countries of origin
print("Top 15 countries:")
print(origins["country"].value_counts().head(15))


## 3. Network Construction

Each network uses the same set of people but defines connections differently.
Connections are weighted using Newman weighting: sharing a credit on a 5-person indie film 
counts more than sharing a credit on a 200-person blockbuster.


In [ ]:
# Load pre-built networks
networks = {}
for name in ["mediascape", "ethnoscape_cross", "ethnoscape_same", "financescape", "ideoscape"]:
    path = DATA_NETWORKS / f"{name}.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        G = nx.node_link_graph(data)
        networks[name] = G
        components = list(nx.connected_components(G))
        largest = max(components, key=len)
        print(f"{name}: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges, {len(components)} components (largest: {len(largest):,})")


### Key structural difference: same-origin fragments into national islands

The cross-origin network is nearly fully connected (one giant component).
The same-origin network fractures into hundreds of separate components — national cinema islands.


## 4. Community Detection & Metrics

Communities are detected using the Leiden algorithm, which finds natural groupings — 
clusters of people who work together more densely than expected by chance.


In [ ]:
# Load pre-computed metrics
metrics = pd.read_csv(DATA_NETWORKS / "metrics.csv")
name_map = people.set_index("nconst")["primaryName"].to_dict()
metrics["name"] = metrics["nconst"].map(name_map)

print("Metrics shape:", metrics.shape)
print("
Columns:", list(metrics.columns))


In [ ]:
# Top 10 most connected people per scape
for scape in ["mediascape", "ethnoscape_cross", "ethnoscape_same"]:
    col = f"{scape}_degree"
    if col in metrics.columns:
        top = metrics.nlargest(10, col)[["name", col]].reset_index(drop=True)
        print(f"
=== Top 10 by {scape} degree ===")
        print(top.to_string(index=False))


## 5. Cross-Network Disjuncture Analysis

The core analytical contribution: measuring where the scapes agree and disagree.


In [ ]:
from scipy.stats import spearmanr

# Centrality correlations across scapes
print("=== Spearman rank correlation of degree centrality ===")
scapes = ["mediascape", "ethnoscape_cross", "ethnoscape_same", "financescape", "ideoscape"]
for i, s1 in enumerate(scapes):
    for s2 in scapes[i+1:]:
        c1, c2 = f"{s1}_degree", f"{s2}_degree"
        if c1 in metrics.columns and c2 in metrics.columns:
            subset = metrics[[c1, c2]].dropna()
            rho, p = spearmanr(subset[c1], subset[c2])
            print(f"  {s1} <-> {s2}: ρ = {rho:.3f} (p = {p:.1e}, n = {len(subset)})")


In [ ]:
from sklearn.metrics import normalized_mutual_info_score

# Community structure comparison (NMI)
print("
=== NMI of community assignments ===")
for i, s1 in enumerate(scapes):
    for s2 in scapes[i+1:]:
        c1, c2 = f"{s1}_community", f"{s2}_community"
        if c1 in metrics.columns and c2 in metrics.columns:
            subset = metrics[[c1, c2]].dropna()
            subset = subset[(subset[c1] >= 0) & (subset[c2] >= 0)]
            if len(subset) > 10:
                nmi = normalized_mutual_info_score(subset[c1], subset[c2])
                print(f"  {s1} <-> {s2}: NMI = {nmi:.3f} (n = {len(subset)})")


### Key finding: NMI of 0.24 between cross-origin and same-origin

This means international and domestic collaboration create **fundamentally different community structures**.
The people grouped together by cross-border work are not the same groups formed by domestic work.


## 6. Outcome Analysis: Does Network Structure Predict Quality?

In [ ]:
# Geographic diversity vs. rating regression
import statsmodels.api as sm

directors = film_people[film_people["category"] == "director"].copy()
dir_films = directors.merge(films[["tconst", "averageRating", "roi", "budget"]], on="tconst")
dir_agg = dir_films.groupby("nconst").agg(
    avg_rating=("averageRating", "mean"),
    avg_roi=("roi", "mean"),
    n_films=("tconst", "nunique"),
    avg_budget=("budget", "mean"),
).reset_index()
dir_agg = dir_agg.merge(metrics[["nconst", "blau_index", "mediascape_clustering"]], on="nconst", how="left")

# Regression: Blau → Rating
subset = dir_agg.dropna(subset=["blau_index", "avg_rating"])
X = sm.add_constant(subset[["blau_index", "n_films"]])
model = sm.OLS(subset["avg_rating"], X).fit(cov_type="HC1")
print("=== Geographic Diversity → Rating ===")
print(f"N = {len(subset)}, R² = {model.rsquared:.4f}")
print(model.summary2().tables[1])


In [ ]:
# Mediation: does budget explain the diversity-rating link?
subset2 = dir_agg.dropna(subset=["blau_index", "avg_rating", "avg_budget"])
subset2["log_budget"] = np.log1p(subset2["avg_budget"])

# Without budget control
X1 = sm.add_constant(subset2[["blau_index"]])
m1 = sm.OLS(subset2["avg_rating"], X1).fit(cov_type="HC1")

# With budget control
X2 = sm.add_constant(subset2[["blau_index", "log_budget"]])
m2 = sm.OLS(subset2["avg_rating"], X2).fit(cov_type="HC1")

print(f"Without budget: β_diversity = {m1.params['blau_index']:.3f} (p = {m1.pvalues['blau_index']:.4f})")
print(f"With budget:    β_diversity = {m2.params['blau_index']:.3f} (p = {m2.pvalues['blau_index']:.4f})")
print(f"
Drop in effect: {(1 - abs(m2.params['blau_index'])/abs(m1.params['blau_index']))*100:.0f}%")
print("Budget mediates most of the diversity-rating relationship.")


In [ ]:
# Writer clustering → Rating
writers = film_people[film_people["category"] == "writer"].copy()
writer_films = writers.merge(films[["tconst", "averageRating", "budget"]], on="tconst")
writer_agg = writer_films.groupby("nconst").agg(
    avg_rating=("averageRating", "mean"),
    n_films=("tconst", "nunique"),
    avg_budget=("budget", "mean"),
).reset_index()
writer_agg = writer_agg.merge(metrics[["nconst", "mediascape_clustering"]], on="nconst", how="left")

subset3 = writer_agg.dropna(subset=["mediascape_clustering", "avg_rating", "avg_budget"])
subset3["log_budget"] = np.log1p(subset3["avg_budget"])
X3 = sm.add_constant(subset3[["mediascape_clustering", "log_budget"]])
m3 = sm.OLS(subset3["avg_rating"], X3).fit(cov_type="HC1")
print("=== Writer Clustering → Rating (controlling for budget) ===")
print(f"N = {len(subset3)}, R² = {m3.rsquared:.4f}")
print(m3.summary2().tables[1])


## 7. Self-Organizing Map

The SOM projects each person's multi-scape profile (18 features) onto a 2D grid,
revealing natural archetypes of film professionals.


In [ ]:
# Load SOM results
with open(DATA_NETWORKS / "som_grid.json") as f:
    som = json.load(f)

grid = np.array(som["populations"])
print(f"Grid size: {som['grid_size']['x']}x{som['grid_size']['y']}")
print(f"Features: {som['features']}")
print(f"Cells occupied: {(grid > 0).sum()}/{grid.size}")
print(f"Max cell population: {grid.max():.0f}")
print(f"Total people mapped: {grid.sum():.0f}")


## 8. Summary of Findings

1. **The diversity paradox is a budget effect.** Geographic diversity correlates with lower ratings, but 61% of this is explained by budget. Diverse casts appear in big international co-productions that score lower critically.

2. **International and domestic cinema are structurally different worlds.** NMI = 0.24 between cross-origin and same-origin networks — the strongest disjuncture in the analysis.

3. **Tight circles make better films.** Writer clustering is the strongest predictor of film quality (β = 7.03, p < 0.001, R² = 0.187).

4. **Financial returns are decoupled from network structure.** No network metric significantly predicts ROI.

5. **Different flows elevate different people.** Cross-origin degree correlates only 0.49 with same-origin degree — who matters internationally is substantially different from who matters domestically.


---
*See the [interactive site](https://moealhassan.github.io/INFO230/MiniProject2/site-v2/) for full visualizations.*

*All source code is in the [](https://github.com/MoeAlhassan/INFO230/tree/main/MiniProject2/src) directory.*
